《Think Stats》 3판은 [Bookshop.org](https://bookshop.org/a/98697/9781098190255)와 [Amazon](https://amzn.to/42lmxwu)에서 구할 수 있습니다. 두 링크는 원저자의 제휴 링크입니다. 책이 도움이 되었다면 Green Tea Press의 [무료 공개 교육 자료 제작을 후원](https://greenteapress.com/support/?book=ts3)하는 것도 생각해 주세요.

# 해석적 방법

이 책에서는 시뮬레이션과 재표집 같은 계산 방법을 중심으로 문제를 풀었습니다. 그중에는 분포의 수학적 성질을 이용해, 시뮬레이션 없이 더 빠르게 답을 구할 수 있는 문제도 있습니다. 이를 해석적 방법이라고 합니다. 이 장에서는 그 원리를 살펴보고 시뮬레이션과 함께 사용하는 방법을 알아보겠습니다.

[원문 노트북을 Colab에서 실행하기](https://colab.research.google.com/github/AllenDowney/ThinkStats/blob/30ecbdcf05723f1190cae2aeaff5848f59f8ca26/nb/chap14.ipynb).

In [1]:
from os.path import basename, exists


def download(url):
    filename = basename(url)
    if not exists(filename):
        from urllib.request import urlretrieve

        local, _ = urlretrieve(url, filename)
        print("Downloaded " + local)


download("https://github.com/AllenDowney/ThinkStats/raw/v3/nb/thinkstats.py")

In [2]:
try:
    import empiricaldist
except ImportError:
    %pip install empiricaldist

In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from thinkstats import decorate

## 정규확률그림

많은 해석적 방법이 정규분포의 성질을 사용합니다. 실제 측정값 중 정규분포로 근사할 수 있는 것이 많고, 정규분포 자체에도 계산하기 좋은 수학적 성질이 있기 때문입니다. 펭귄 자료의 측정값부터 살펴본 다음 그 성질들을 알아보겠습니다.

펭귄 자료 출처: Horst, Hill, Gorman(2020), *palmerpenguins*, [DOI: 10.5281/zenodo.3960218](https://doi.org/10.5281/zenodo.3960218). 원래 연구는 Gorman, Williams, Fraser(2014), *PLOS ONE* 9(3), e90081, [DOI: 10.1371/journal.pone.0090081](https://doi.org/10.1371/journal.pone.0090081)입니다.

In [4]:
download(
    "https://raw.githubusercontent.com/allisonhorst/palmerpenguins/c19a904462482430170bfe2c718775ddb7dbb885/inst/extdata/penguins_raw.csv"
)

CSV를 읽습니다.

In [5]:
penguins = pd.read_csv("penguins_raw.csv")
penguins.shape

세 종 중 아델리펭귄을 선택합니다.

In [6]:
adelie = penguins.query('Species.str.startswith("Adelie")').copy()
len(adelie)

체중이 정규분포에 가까운지 보기 위해 경험적 CDF를 구합니다.

In [7]:
from empiricaldist import Cdf

weights = adelie["Body Mass (g)"].dropna()
cdf_weights = Cdf.from_seq(weights)

같은 평균과 표준편차를 가진 정규분포의 이론 CDF도 구합니다.

In [8]:
m, s = weights.mean(), weights.std()
m, s

In [9]:
from scipy.stats import norm

dist = norm(m, s)
qs = np.linspace(m - 3.5 * s, m + 3.5 * s)
ps = dist.cdf(qs)

자료와 모형의 CDF를 비교합니다.

In [10]:
model_options = dict(color="gray", alpha=0.5, label="model")
plt.plot(qs, ps, **model_options)
cdf_weights.plot(label="data")

decorate(ylabel="CDF")

정규분포가 쓸 만한 근사일 수 있지만 완벽히 맞지는 않습니다. CDF 비교는 좋은 진단 방법이지만, 모형의 평균과 표준편차를 얼마나 잘 추정했는지에도 영향을 받습니다.

다른 방법인 **정규확률그림**은 특정 평균·표준편차를 먼저 맞추기보다 점들이 직선에 가까운지 봅니다. y축에는 정렬한 관측값을 놓습니다.

In [11]:
ys = np.sort(weights)

x축에는 대응하는 표준정규분포의 분위수를 놓습니다. `norm.ppf`는 inverse CDF, 즉 역누적분포함수입니다.

In [12]:
n = len(weights)
ps = (np.arange(n) + 0.5) / n
xs = norm.ppf(ps)

자료가 정규분포에서 나왔다면 점들은 대체로 직선 위에 놓입니다. `linregress`로 직선을 맞춰 보겠습니다.

In [13]:
from scipy.stats import linregress

results = linregress(xs, ys)
intercept, slope = results.intercept, results.slope

fit_xs = np.linspace(-3, 3)
fit_ys = intercept + slope * fit_xs

분위수와 관측값, 적합 직선을 함께 그립니다.

In [14]:
plt.plot(fit_xs, fit_ys, **model_options)
plt.plot(xs, ys, label="data")

decorate(xlabel="Standard normal", ylabel="Body mass (g)")

완벽한 직선은 아닙니다. 체중이 다른 암수 집단을 합쳤기 때문일 수 있습니다. 집단을 나눠 그리려고 앞의 과정을 함수로 묶겠습니다.

In [15]:
def normal_probability_plot(sample, **options):
    """Makes a normal probability plot with a fitted line."""
    n = len(sample)
    ps = (np.arange(n) + 0.5) / n
    xs = norm.ppf(ps)
    ys = np.sort(sample)

    results = linregress(xs, ys)
    intercept, slope = results.intercept, results.slope

    fit_xs = np.linspace(-3, 3)
    fit_ys = intercept + slope * fit_xs

    plt.plot(fit_xs, fit_ys, color="gray", alpha=0.5)
    plt.plot(xs, ys, **options)
    decorate(xlabel="Standard normal")

암컷과 수컷을 따로 그립니다.

In [16]:
grouped = adelie.groupby("Sex")

weights_male = grouped.get_group("MALE")["Body Mass (g)"]
normal_probability_plot(weights_male, ls="--", label="Male")

weights_female = grouped.get_group("FEMALE")["Body Mass (g)"]
normal_probability_plot(weights_female, label="Female")

decorate(ylabel="Weight (g)")

각 집단의 점들은 더 직선에 가깝습니다. 서로 다른 평균의 두 정규분포를 섞으면 전체 분포가 정규분포가 아닐 수 있습니다. 이제 정규분포의 계산상 성질을 살펴보겠습니다.

> 번역자 주: 정규확률그림은 적합을 살피는 진단입니다. 직선에 가깝다는 이유만으로 정확한 정규성을 증명하지는 않습니다. 아래 그림과 실행창은 관측 분위수와 같은 단위의 정규 기준 분위수를 비교하는 Q–Q 형태를 사용합니다. 축의 위치·척도를 바꾼 표현이지만 곡률을 보는 원리는 같습니다.

## 정규분포

정규분포를 나타내는 클래스를 정의합니다. `mu`는 평균, `sigma2`는 분산입니다. 표준편차인 sigma의 제곱이라는 뜻에서 이름에 2를 붙였습니다.

In [17]:
class Normal:
    """Represents a Normal distribution"""

    def __init__(self, mu, sigma2):
        """Make a Normal object.

        mu: mean
        sigma2: variance
        """
        self.mu = mu
        self.sigma2 = sigma2

    def __repr__(self):
        """Returns a string representation."""
        return f"Normal({self.mu}, {self.sigma2})"

    __str__ = __repr__

수컷 체중의 평균·분산으로 `Normal` 객체를 만듭니다.

In [18]:
m, s = weights_male.mean(), weights_male.std()
dist_male = Normal(m, s**2)
dist_male

암컷 체중에 대해서도 만듭니다.

In [19]:
m, s = weights_female.mean(), weights_female.std()
dist_female = Normal(m, s**2)
dist_female

정규분포의 표본을 생성하는 `sample` 메서드를 추가합니다. 원문은 Jupyter 전용 `add_method_to` 매직을 사용합니다. 본문 셀에서는 같은 함수를 정의한 뒤 클래스에 대입하는 일반적인 Python 코드로 바꾸었습니다.

In [20]:
%%add_method_to Normal


def sample(self, n):
    """Generate a random sample from this distribution."""
    sigma = np.sqrt(self.sigma2)
    return np.random.normal(self.mu, sigma, n)

첫 번째 성질은 독립적인 두 정규변수를 더한 값도 정규분포를 따른다는 것입니다. 두 체중 모형에서 각각 표본을 만들어 더한 뒤 정규확률그림을 그리겠습니다.

In [21]:
sample_sum = dist_male.sample(1000) + dist_female.sample(1000)
normal_probability_plot(sample_sum)

decorate(ylabel="Total weight (g)")

점들이 직선에 가깝습니다. 합의 평균과 분산은 두 모형의 모수만으로도 구할 수 있습니다.

In [22]:
%%add_method_to Normal


def __add__(self, other):
    """Distribution of the sum of two normal distributions."""
    return Normal(self.mu + other.mu, self.sigma2 + other.sigma2)

독립인 두 변수의 합에서는 평균끼리 더하고 분산끼리 더합니다. `__add__`를 정의했으므로 `+`로 합의 분포를 구할 수 있습니다.

In [23]:
dist_sum = dist_male + dist_female
dist_sum

정규분포의 이론 CDF를 그리는 메서드를 추가해 결과를 확인하겠습니다.

In [24]:
%%add_method_to Normal


def plot_cdf(self, n_sigmas=3.5, **options):
    """Plot the CDF of this distribution."""
    mu, sigma = self.mu, np.sqrt(self.sigma2)
    low, high = mu - n_sigmas * sigma, mu + n_sigmas * sigma
    xs = np.linspace(low, high, 101)
    ys = norm.cdf(xs, mu, sigma)
    plt.plot(xs, ys, **options)

가상 표본 합의 경험적 CDF와 이론 CDF를 비교합니다.

In [25]:
dist_sum.plot_cdf(**model_options)
Cdf.from_seq(sample_sum).plot(label="sample")

decorate(xlabel="Total weight (g)", ylabel="CDF")

두 결과가 잘 맞습니다. 같은 정규분포에서 독립적으로 n개를 뽑아 더해도 정규분포입니다. 예로 수컷 체중 73개를 더하는 실험을 1001번 합니다.

In [26]:
n = len(weights_male)
sample_sums_male = [dist_male.sample(n).sum() for i in range(1001)]
n

n개 합의 모형은 평균과 분산을 각각 n배 해 만듭니다.

In [27]:
%%add_method_to Normal


def sum(self, n):
    """Return the distribution of the sum of n values."""
    return Normal(n * self.mu, n * self.sigma2)

체중 합의 정규분포 모형입니다.

In [28]:
dist_sums_male = dist_male.sum(n)

시뮬레이션의 경험적 분포와 비교합니다.

In [29]:
dist_sums_male.plot_cdf(**model_options)
Cdf.from_seq(sample_sums_male).plot(label="sample")

decorate(xlabel="Total weights (g)", ylabel="CDF")

모형이 표본 합의 분포에 잘 맞습니다. 이처럼 독립인 측정값 n개의 합을 매번 시뮬레이션하지 않고도 분포를 구할 수 있습니다.

> 번역자 주: 분산을 단순히 더하는 계산에는 독립성 또는 적어도 공분산 0이라는 조건이 필요합니다. 일반적으로 $\operatorname{Var}(X+Y)=\operatorname{Var}(X)+\operatorname{Var}(Y)+2\operatorname{Cov}(X,Y)$입니다. 정규분포의 합이 정규라는 성질에도 독립 정규변수 또는 적절한 결합정규 가정이 들어 있습니다.

## 표본평균의 분포

표본 합의 분포를 알면 평균의 분포도 구할 수 있습니다. 정규변수에 상수를 곱하거나 나누어도 정규분포라는 세 번째 성질을 사용합니다.

In [30]:
%%add_method_to Normal


def __mul__(self, factor):
    """Multiplies by a scalar."""
    return Normal(factor * self.mu, factor**2 * self.sigma2)

In [31]:
%%add_method_to Normal


def __truediv__(self, factor):
    """Divides by a scalar."""
    return self * (1 / factor)

상수를 곱하면 평균에는 그 상수를, 분산에는 상수의 제곱을 곱합니다. 합의 분포를 n으로 나누어 표본평균 분포를 구하겠습니다.

In [32]:
dist_mean_male = dist_sums_male / n

같은 실험의 표본평균들도 계산합니다.

In [33]:
sample_means_male = np.array(sample_sums_male) / n

경험적 CDF와 정규 모형을 비교합니다.

In [34]:
dist_mean_male.plot_cdf(**model_options)
Cdf.from_seq(sample_means_male).plot(label="sample")

decorate(xlabel="Average weight (g)", ylabel="CDF")

두 결과가 잘 맞습니다. 해석적 계산은 재표집보다 훨씬 빠릅니다. 표본평균의 표준오차는 이 표집분포의 표준편차입니다.

In [35]:
standard_error = np.sqrt(dist_mean_male.sigma2)
standard_error

앞 계산에서는 분산에 n을 곱한 뒤 n²으로 나누었습니다. 결과적으로 분산은 원래의 1/n, 표준편차는 1/√n이 됩니다.

$$\operatorname{Var}(\bar X)=\frac{\sigma^2}{n},\qquad \operatorname{SE}(\bar X)=\frac{\sigma}{\sqrt n}$$

이 공식을 직접 사용해 표준오차를 구할 수 있습니다. σ를 표본 표준편차로 바꾸면 표준오차의 추정값이 됩니다. 독립성 같은 전제를 잊지 마세요.

In [36]:
standard_error = weights_male.std() / np.sqrt(n)
standard_error

이번에는 두 집단의 평균 차이 분포를 구하겠습니다.

## 차이의 분포

암컷 체중의 표본평균 분포도 같은 방법으로 구합니다.

In [37]:
n = len(weights_female)
dist_mean_female = dist_female.sum(n) / n
dist_mean_female

암수 평균의 표집분포 두 개에서 차이의 분포를 만들겠습니다.

In [38]:
%%add_method_to Normal


def __sub__(self, other):
    """Compute the distribution of a difference."""
    return Normal(self.mu - other.mu, self.sigma2 + other.sigma2)

차이의 평균은 평균의 차이지만, 독립인 두 변수의 **분산은 뺄 때도 더합니다**. 두 번째 변수의 부호를 바꾸어도 분산은 같고, 첫 번째 변수와 더할 때 두 분산이 합쳐지기 때문입니다. 해석적으로 구한 분포를 보겠습니다.

In [39]:
dist_diff_means = dist_mean_male - dist_mean_female
dist_diff_means

모형에서 난수 차이도 생성합니다.

In [40]:
sample_sums_female = [dist_female.sample(n).sum() for i in range(1001)]
sample_means_female = np.array(sample_sums_female) / n
sample_diff_means = sample_means_male - sample_means_female

경험적 CDF와 이론 CDF를 비교합니다.

In [41]:
dist_diff_means.plot_cdf(**model_options)
Cdf.from_seq(sample_diff_means).plot(label="sample")

decorate(xlabel="Difference in average weight (g)", ylabel="CDF")

잘 맞습니다. 역 CDF를 구하는 메서드를 추가하면 평균 차이의 구간도 계산할 수 있습니다.

In [42]:
%%add_method_to Normal


def ppf(self, xs):
    sigma = np.sqrt(self.sigma2)
    return norm.ppf(xs, self.mu, sigma)

5·95백분위수로 90% 근사 신뢰구간을 구합니다.

In [43]:
ci90 = dist_diff_means.ppf([0.05, 0.95])
ci90

가상 표본의 백분위수에서도 비슷한 값을 얻습니다.

In [44]:
np.percentile(sample_diff_means, [5, 95])

해석적 결과는 빠르고 난수에 의존하지 않습니다. 하지만 지금까지는 측정값의 분포가 정규라는 가정을 사용했습니다. 실제 자료는 정확한 정규분포가 아닙니다. 그래도 여러 측정값을 더하면 합의 분포가 정규에 가까워지는 경우가 많습니다. 이를 설명하는 것이 중심극한정리입니다.

> 번역자 주: 차이의 분산은 일반적으로 두 분산의 합에서 공분산의 두 배를 뺀 값입니다. 같은 대상을 전후로 측정한 짝 자료에 서로 독립인 두 집단 공식을 그대로 쓰면 안 됩니다. 표본에서 추정한 모수로 구한 정규 구간은 모수를 알고 있을 때 구한 정확한 구간과도 구별해야 합니다.

## 중심극한정리

정규분포에서 뽑은 독립 변수의 합은 정규분포지만, 대부분의 분포가 이런 닫힘 성질을 갖지는 않습니다. 지수변수의 합은 지수분포가 아닙니다.

그럼에도 유한한 평균 μ와 양의 유한한 분산 σ²를 가진 같은 분포에서 독립적으로 뽑은 값들에서는, n이 커질수록 **표준화한 합**이 표준정규분포에 가까워집니다.

$$\frac{\sum_{i=1}^{n}X_i-n\mu}{\sigma\sqrt n}\ \xrightarrow{d}\ N(0,1)$$

따라서 합의 분포는 평균 nμ·분산 nσ²인 정규분포로, 표본평균의 분포는 평균 μ·분산 σ²/n인 정규분포로 근사할 수 있습니다.

이것이 **중심극한정리**(CLT)입니다. 유용하지만 조건이 있습니다.

- 같은 분포에서 나와야 합니다. 더 일반적인 정리는 이 조건을 완화합니다.
- 위의 기본 형태에서는 독립이어야 합니다. 의존 자료에는 별도의 조건과 분산 계산이 필요합니다.
- 모평균과 모분산이 유한해야 합니다. 꼬리가 너무 두꺼운 일부 분포는 해당하지 않습니다.

생물의 여러 특성은 많은 유전·환경 요인의 작은 기여가 더해져 나타납니다. 중심극한정리는 이런 상황에서 정규 모양이 자주 나타나는 이유를 설명합니다. 언제 잘 작동하고 언제 잘 작동하지 않는지 실험하겠습니다. 먼저 지수분포에서 n개를 더하는 실험을 크기별로 1001번 합니다.

In [45]:
np.random.seed(17)

In [46]:
lam = 1
df_sample_expo = pd.DataFrame()
for n in [1, 10, 100]:
    df_sample_expo[n] = [np.sum(np.random.exponential(lam, n)) for _ in range(1001)]

각 크기에서 합의 평균을 확인합니다.

In [47]:
df_sample_expo.mean()

원래 평균이 1이므로 10개 합의 평균은 약 10, 100개 합의 평균은 약 100입니다.

다음 함수는 각 크기의 합에 대해 정규확률그림을 그립니다.

In [48]:
def normal_plot_samples(df_sample, ylabel=""):
    """Normal probability plots for samples of sums."""
    plt.figure(figsize=(6.8, 2.6))
    for i, n in enumerate(df_sample):
        plt.subplot(1, 3, i + 1)
        normal_probability_plot(df_sample[n])
        decorate(
            title="n=%d" % n,
            xticks=[],
            yticks=[],
            xlabel="Standard normal",
            ylabel=ylabel,
        )

세 크기의 결과를 비교합니다.

In [49]:
normal_plot_samples(df_sample_expo, ylabel="Sum of exponential values")

n=1에서는 지수분포라 직선이 아니지만, n=10에서는 정규에 가까워지고 n=100에서는 아주 비슷해집니다. 원래 분포의 치우침이 작으면 적은 n에서도 근사가 좋아질 수 있고, 크면 더 오래 걸립니다. 로그정규분포의 합을 보겠습니다.

In [50]:
mu, sigma = 3.0, 1.0
df_sample_lognormal = pd.DataFrame()
for n in [1, 10, 100]:
    df_sample_lognormal[n] = [
        np.sum(np.random.lognormal(mu, sigma, n)) for _ in range(1001)
    ]

같은 표본 크기들의 정규확률그림을 그립니다.

In [51]:
normal_plot_samples(df_sample_lognormal, ylabel="Sum of lognormal values")

n=1과 10에서는 정규 근사가 나쁘고, 100에서도 꼬리가 다릅니다. 로그정규분포는 평균·분산이 유한하므로 결국 수렴하지만, 치우침이 크면 실용적인 표본 크기로는 부족할 수 있습니다. 어떤 분포는 아예 이 정리의 조건을 만족하지 않습니다.

## 중심극한정리의 한계

파레토분포는 모수에 따라 평균이나 분산이 유한하지 않을 수 있습니다. α=1인 파레토분포는 유한한 평균과 분산이 없어 위의 CLT를 적용할 수 없습니다.

In [52]:
alpha = 1.0
df_sample = pd.DataFrame()
for n in [1, 10, 100]:
    df_sample[n] = [np.sum(np.random.pareto(alpha, n)) for _ in range(1001)]

크기별 합의 정규확률그림을 그립니다.

In [53]:
normal_plot_samples(df_sample, ylabel="Sum of Pareto values")

n=100이어도 합의 분포는 정규분포와 크게 다릅니다. 의존성도 문제가 됩니다. 다음에는 상관된 정규 시퀀스를 만든 뒤 지수 시퀀스로 바꾸는 `generate_expo_correlated`를 사용하겠습니다.

In [54]:
def generate_normal_correlated(n, rho):
    """Generates an array of correlated values from a standard normal dist."""
    xs = np.empty(n)
    xs[0] = np.random.normal(0, 1)

    sigma = np.sqrt(1 - rho**2)
    for i in range(1, n):
        xs[i] = rho * xs[i - 1] + np.random.normal(0, sigma)

    return xs

상관된 정규값들을 지수값들로 바꾸는 함수입니다.

In [55]:
from scipy.stats import expon


def generate_expo_correlated(n, rho):
    """Generates a sequence of correlated values from an exponential dist."""
    normal = generate_normal_correlated(n, rho)
    uniform = norm.cdf(normal)
    expo = expon.ppf(uniform)
    return expo

정규 CDF로 0~1의 균등 주변분포 값으로 바꾸고, 지수 역 CDF를 적용합니다. 주변분포는 지수분포가 되어도 시퀀스의 의존성은 남습니다.

각 표본 크기에서 합을 1001개 생성합니다.

In [56]:
rho = 0.8
df_sample = pd.DataFrame()
for n in [1, 10, 100]:
    df_sample[n] = [np.sum(generate_expo_correlated(n, rho)) for _ in range(1001)]

그 합의 정규확률그림을 그립니다.

In [57]:
normal_plot_samples(df_sample, ylabel="Sum of correlated values")

원문에서 ρ=0.8로 생성한 시퀀스는 강한 의존성을 가지며 수렴도 느립니다. 먼 시점들 사이의 의존성까지 강하게 남으면 보통의 정규 근사가 성립하지 않을 수도 있습니다.

> 번역자 주: 여기서 ρ는 먼저 생성한 정규 시퀀스의 상관 모수입니다. CDF와 역 CDF의 비선형 변환 뒤 지수값들의 피어슨 상관이 정확히 ρ로 유지되는 것은 아닙니다. 독립이 깨졌다고 모든 종류의 CLT가 실패하는 것도 아닙니다. 적절한 의존성 조건에서는 정규 근사가 가능하지만, 분산은 자기공분산 항들을 포함해야 합니다.

## 중심극한정리 활용하기

[9장](/posts/statistics/think-stats-09#평균-차이-검정하기)의 첫아이와 다른 아이의 평균 임신 기간 차이로 돌아가겠습니다. 같은 NSFG 자료를 사용합니다.

자료를 가져오는 방법은 노트북에 있습니다.

In [58]:
download("https://github.com/AllenDowney/ThinkStats/raw/v3/nb/nsfg.py")
download("https://github.com/AllenDowney/ThinkStats/raw/v3/data/2002FemPreg.dct")
download("https://github.com/AllenDowney/ThinkStats/raw/v3/data/2002FemPreg.dat.gz")

생아 출산을 첫아이와 다른 아이로 나눕니다.

In [59]:
from nsfg import get_nsfg_groups

live, firsts, others = get_nsfg_groups()

평균 차이는 약 0.078주입니다.

In [60]:
delta = firsts["prglngth"].mean() - others["prglngth"].mean()
delta

두 집단의 평균·분산이 같다는 귀무가설 아래, 전체 생아 출산에서 공통 모수를 추정합니다.

In [61]:
all_lengths = live["prglngth"]
m, s2 = all_lengths.mean(), all_lengths.var()

임신 기간 자체는 정규분포가 아닙니다. 그래도 표본평균의 분포는 정규로 근사할 수 있습니다. 다음 함수는 자료의 평균·분산과 표본 크기로 평균의 표집분포 `Normal`을 만듭니다.

In [62]:
def sampling_dist_mean(data, n):
    mean, var = data.mean(), data.var()
    dist = Normal(mean, var)
    return dist.sum(n) / n

귀무가설 아래 첫아이 집단의 평균 임신 기간에 대한 정규 근사입니다.

In [63]:
n1 = firsts["totalwgt_lb"].count()
dist_firsts = sampling_dist_mean(all_lengths, n1)
n1

다른 아이 집단도 구합니다.

In [64]:
n2 = others["totalwgt_lb"].count()
dist_others = sampling_dist_mean(all_lengths, n2)
n2

두 평균 차이의 표집분포를 만듭니다.

In [65]:
dist_diff = dist_firsts - dist_others
dist_diff

귀무모형에서는 평균 차이의 기댓값이 0입니다. 차이의 표집분산은 약 0.0032주²로, 우연한 변동의 크기를 나타냅니다. 재표집 분포와 비교하겠습니다.

In [66]:
sample_firsts = [np.random.choice(all_lengths, n1).mean() for i in range(1001)]
sample_others = [np.random.choice(all_lengths, n2).mean() for i in range(1001)]
sample_diffs = np.subtract(sample_firsts, sample_others)

관측된 차이의 양·음 위치를 수직선으로 표시해 정규 모형과 경험적 CDF를 비교합니다.

In [67]:
dist_diff.plot_cdf(**model_options)
Cdf.from_seq(sample_diffs).plot(label="sample")
plt.axvline(delta, ls=":")
plt.axvline(-delta, ls=":")

decorate(xlabel="Difference in pregnancy length", ylabel="CDF")

집단 크기가 크고 원자료의 치우침도 극단적이지 않아 정규 근사가 잘 맞습니다. 정규 CDF로 꼬리 확률을 구하겠습니다.

In [68]:
%%add_method_to Normal


def cdf(self, xs):
    sigma = np.sqrt(self.sigma2)
    return norm.cdf(xs, self.mu, sigma)

귀무모형에서 양의 차이 δ 이상일 오른쪽 꼬리 확률입니다.

In [69]:
right = 1 - dist_diff.cdf(delta)
right

−δ 이하일 왼쪽 꼬리 확률도 구합니다.

In [70]:
left = dist_diff.cdf(-delta)
left

정규분포가 대칭이므로 양쪽 확률이 같습니다. 두 꼬리를 더하면 차이의 절댓값이 δ 이상일 확률입니다.

In [71]:
left + right

p값은 약 0.170으로 [9장](/posts/statistics/think-stats-09#평균-차이-검정하기)의 재표집 결과와 비슷합니다. SciPy의 `ttest_ind`도 서로 독립인 두 집단의 평균 차이를 검정합니다.

In [72]:
from scipy.stats import ttest_ind

result = ttest_ind(firsts["prglngth"], others["prglngth"])
result.pvalue

표본이 크면 t검정 결과는 정규 근사와 비슷합니다. t검정은 정규분포 대신 **t분포**를 사용합니다. 다음에는 상관 검정에도 t분포를 사용하겠습니다.

> 번역자 주: 기본 `ttest_ind`는 같은 분산을 가정합니다. `equal_var=False`의 웰치 검정은 그 가정을 완화합니다. 독립 관측·표본 설계 같은 조건은 여전히 필요합니다. 정규 근사나 재표집이 같은 값에 가까워졌다고 표집 편향까지 해결된 것은 아닙니다.

## 상관 검정

[9장](/posts/statistics/think-stats-09#상관-검정하기)에서는 산모 나이와 출생 체중의 상관을 순열로 검정했고, 매우 작은 꼬리 확률을 보았습니다. 이번에는 해석적으로 계산하겠습니다.

서로 독립인 정규변수의 크기 n인 표본쌍에서 피어슨 상관 r을 구하면, 다음 변환을 사용할 수 있습니다.

$$t=r\sqrt{\frac{n-2}{1-r^2}}$$

In [73]:
def transform_correlation(r, n):
    return r * np.sqrt((n - 2) / (1 - r**2))

귀무가설 아래 이 통계량은 자유도 n−2의 t분포를 따릅니다. 독립 표준정규 표본을 생성해 확인하겠습니다.

In [74]:
def generate_data(n):
    """Uncorrelated sequences from a standard normal."""
    xs = np.random.normal(0, 1, n)
    ys = np.random.normal(0, 1, n)
    return xs, ys

생성한 두 표본의 상관을 구하는 함수입니다.

In [75]:
def correlation(data):
    xs, ys = data
    return np.corrcoef(xs, ys)[0, 1]

여러 표본쌍을 생성하고 상관을 모읍니다.

In [76]:
n = 100
rs = [correlation(generate_data(n)) for i in range(1001)]

상관들을 t통계량으로 바꿉니다.

In [77]:
ts = transform_correlation(np.array(rs), n)

t분포 CDF를 만드는 함수로 비교합니다.

In [78]:
from scipy.stats import t as student_t


def make_student_cdf(df):
    """Computes the CDF of a Student t distribution."""
    ts = np.linspace(-3, 3, 101)
    ps = student_t.cdf(ts, df=df)
    return Cdf(ps, ts)

`df`는 degrees of freedom, 즉 **자유도**입니다. 자유도 n−2의 이론 CDF와 경험적 CDF를 함께 그립니다.

In [79]:
make_student_cdf(df=n - 2).plot(**model_options)

cdf_ts = Cdf.from_seq(ts)
cdf_ts.plot(label="random normals")

decorate(xlabel="Transformed correlation", ylabel="CDF")

독립 정규표본의 변환 상관은 t분포와 잘 맞습니다. 정규가 아닌 자료에서도 적절한 조건에서 큰 표본 근사가 유용할 수 있지만, 모든 무상관 자료에 같은 검정이 타당한 것은 아닙니다. 산모 나이와 출생 체중이 모두 있는 행을 고르겠습니다.

In [80]:
valid = live.dropna(subset=["agepreg", "totalwgt_lb"])
n = len(valid)
n

관측 상관은 약 0.07입니다.

In [81]:
data = valid["agepreg"].values, valid["totalwgt_lb"].values
r_actual = correlation(data)
r_actual

앞 장처럼 한 변수의 순서를 섞어 귀무모형을 구현합니다.

In [82]:
def permute(data):
    """Shuffle the x values."""
    xs, ys = data
    new_xs = xs.copy()
    np.random.shuffle(new_xs)
    return new_xs, ys

여러 순열의 상관을 모읍니다.

In [83]:
permuted_corrs = [correlation(permute(data)) for i in range(1001)]

그 상관을 t통계량으로 바꿉니다.

In [84]:
ts = transform_correlation(np.array(permuted_corrs), n)

경험적 CDF와 자유도 n−2의 t분포 CDF를 비교합니다.

In [85]:
make_student_cdf(n - 2).plot(**model_options)
Cdf.from_seq(ts).plot(label="permuted data")

decorate(xlabel="Transformed correlation", ylabel="CDF")

이 자료에서는 근사가 잘 맞습니다. 관측 상관도 같은 방식으로 변환하겠습니다.

In [86]:
t_actual = transform_correlation(r_actual, n)

귀무모형에서 관측 t통계량 이상일 확률을 계산합니다.

In [87]:
right = 1 - student_t.cdf(t_actual, df=n - 2)
right

반대쪽 꼬리의 확률도 계산합니다.

In [88]:
left = student_t.cdf(-t_actual, df=n - 2)
left

두 값을 더하면 양측 p값입니다.

In [89]:
left + right

SciPy의 `pearsonr`도 상관과 p값을 반환합니다.

In [90]:
from scipy.stats import pearsonr

corr, p_value = pearsonr(*data)
p_value

두 결과는 거의 같습니다. 순열을 적은 횟수만 수행하면 매우 작은 p값의 자릿수를 구별하기 어렵지만, 해석적 방법으로는 빠르게 계산할 수 있습니다. 다만 0.001보다 훨씬 작은 p값을 더 정밀하게 구하는 것보다, 효과의 크기와 자료·모형의 한계를 확인하는 편이 실용적으로 중요할 때가 많습니다.

## 카이제곱 검정

[9장](/posts/statistics/think-stats-09#비율-검정하기)의 주사위 관측 도수를 다시 사용합니다.

In [91]:
from empiricaldist import Hist

qs = np.arange(1, 7)
freqs = [8, 9, 19, 5, 8, 11]
observed = Hist(freqs, qs)
observed.index.name = "outcome"
observed

각 눈의 기대 도수를 구합니다.

In [92]:
num_rolls = observed.sum()
outcomes = observed.qs
expected = Hist(num_rolls / 6, outcomes)

카이제곱 통계량을 계산합니다.

In [93]:
def chi_squared_stat(observed, expected):
    diffs = (observed - expected) ** 2
    ratios = diffs / expected
    return np.sum(ratios.values.flatten())

In [94]:
observed_chi2 = chi_squared_stat(observed, expected)

카이제곱 통계량이 널리 쓰이는 이유 중 하나는 귀무분포를 효율적으로 근사할 수 있기 때문입니다. 그 분포도 **카이제곱분포**라고 부릅니다. 공정한 주사위 모의실험을 만들겠습니다.

In [95]:
def simulate_dice(observed):
    n = np.sum(observed)
    rolls = np.random.choice(observed.qs, n, replace=True)
    hist = Hist.from_seq(rolls)
    return hist

여러 번 실험해 각 도수표의 카이제곱 통계량을 구합니다.

In [96]:
simulated_chi_squared = [
    chi_squared_stat(simulate_dice(observed), expected) for i in range(1001)
]
cdf_simulated = Cdf.from_seq(simulated_chi_squared)

자유도 df의 카이제곱 CDF를 계산합니다.

In [97]:
from scipy.stats import chi2 as chi2_dist


def chi_squared_cdf(df):
    """Discrete approximation of the chi-squared CDF."""
    xs = np.linspace(0, 21, 101)
    ps = chi2_dist.cdf(xs, df=df)
    return Cdf(ps, xs)

모든 범주 확률이 미리 주어져 있고 가능한 눈이 n개라면, 큰 표본에서 통계량은 자유도 n−1의 카이제곱분포로 근사됩니다. 총도수가 고정되어 제약 하나가 생깁니다.

In [98]:
n = len(observed)
cdf_model = chi_squared_cdf(df=n - 1)

경험적 CDF와 이론 CDF를 비교합니다.

In [99]:
cdf_model.plot(**model_options)
cdf_simulated.plot(label="simulation")

decorate(xlabel="Chi-squared statistic", ylabel="CDF")

이 예에서는 근사가 비교적 잘 맞습니다. 관측 통계량 이상의 오른쪽 꼬리 확률을 구합니다.

In [100]:
p_value = 1 - chi2_dist.cdf(observed_chi2, df=n - 1)
p_value

SciPy의 `chisquare`로도 구할 수 있습니다.

In [101]:
from scipy.stats import chisquare

chi2_stat, p_value = chisquare(f_obs=observed, f_exp=expected)

같은 p값이 나옵니다.

In [102]:
p_value

카이제곱 통계량은 귀무분포를 빠르게 계산할 수 있다는 장점이 있습니다. 하지만 통계량이 현재 문제에서 중요한 차이를 잘 나타내는지는 별개의 질문입니다.

> 번역자 주: 범주가 드물어 기대 도수가 작으면 이 근사가 부정확할 수 있습니다. 총합과 범주를 보존한 다항 모의실험으로 확인할 수 있습니다. 또한 기대 확률의 모수를 자료에서 추정했다면 자유도도 그에 맞게 조정해야 합니다.

## 시뮬레이션과 해석적 방법 함께 쓰기

이 책은 재표집과 순열 같은 계산 방법을 중심으로 구성했습니다. 원저자는 다음 장점을 듭니다.

- **설명하고 이해하기 쉽습니다.** 특히 p값은 공식을 먼저 배우면 뜻을 놓치기 쉽습니다. 귀무모형에서 자료를 만들고 통계량을 비교하면 무엇의 확률인지 눈으로 확인할 수 있습니다.
- **유연하게 확장할 수 있습니다.** 닫힌 형태의 공식을 구하기 어려운 통계량에도 적용할 수 있고, 문제의 구조에 맞춰 모의실험을 바꿀 수 있습니다.
- **단계별 확인이 가능합니다.** 숫자만 함수에 넣고 결과를 받는 대신, 자료 생성·통계량·꼬리 비율을 나눠 검증할 수 있습니다.

계산 방법에도 독립성·교환 가능성·표집 설계 같은 가정이 있다는 점은 기억해야 합니다.

단점도 있습니다. 계산이 느릴 수 있고, 난수를 쓰면 실행마다 결과가 달라져 비교가 어려울 수 있습니다. 원저자는 다음 흐름을 권합니다.

1. 탐색 단계에서 계산 방법을 사용합니다. 답이 충분하고 실행시간도 괜찮다면 여기서 끝내도 됩니다.
2. 너무 느리다면 최적화할 부분을 찾습니다. 해석적 방법도 최적화의 한 선택지입니다.
3. 계산 방법을 해석적 방법으로 바꾸었다면 둘을 비교해 서로 검증합니다.

많은 실제 문제에서는 첫 단계만으로 충분합니다.

## 용어 정리

- **정규확률그림**: 관측값의 분위수와 정규 기준 분위수를 비교하는 그림.
- **독립표본 t검정**: 독립인 두 집단의 평균 차이에 대한 p값을 계산하는 방법.
- **t분포**: 정규 모형에서 표준화한 평균 차이나 변환된 상관의 귀무분포에 등장하는 분포.
- **카이제곱분포**: 범주 도수의 카이제곱 통계량을 근사하는 분포.
- **카이제곱 통계량**: 관측 도수와 기대 도수의 차이를 기대 크기에 맞춰 제곱해 합친 통계량.

## 연습문제

### 연습문제 14.1

체중 대신 암수 펭귄의 날개 길이를 비교하세요. 관측 평균 차이는 약 4.6mm입니다.

In [103]:
grouped = adelie.groupby("Sex")

lengths_male = grouped.get_group("MALE")["Flipper Length (mm)"]
lengths_female = grouped.get_group("FEMALE")["Flipper Length (mm)"]
observed_diff = lengths_male.mean() - lengths_female.mean()
observed_diff

집단 크기가 서로 다를 수 있다는 점을 반영해 `sampling_dist_mean`으로 두 표본평균의 정규 모형을 만드세요. 차이의 분포와 90% 신뢰구간을 구하세요.

In [104]:
# Solution goes here

In [105]:
# Solution goes here

In [106]:
# Solution goes here

In [107]:
# Solution goes here

In [108]:
# Solution goes here

### 연습문제 14.2

이번에는 출생 체중과 아버지 나이의 상관을 분석하세요. 아버지 나이는 `hpagelb` 열에 있습니다.

In [109]:
valid = live.dropna(subset=["hpagelb", "totalwgt_lb"])
n = len(valid)
n

관측 상관은 약 0.065입니다.

In [110]:
data = valid["hpagelb"].values, valid["totalwgt_lb"].values
r_actual = correlation(data)
r_actual

상관을 t통계량으로 바꾸고 t분포 CDF로 양측 p값을 구하세요. `pearsonr`와 비교하고, 통계적 유의성과 효과 크기를 구별해 해석하세요.

In [111]:
# Solution goes here

In [112]:
# Solution goes here

In [113]:
# Solution goes here

### 연습문제 14.3

[11장](/posts/statistics/think-stats-11#연습문제-112)의 트리버스–윌러드 가설은 모체 상태와 성비의 관계를 제안합니다. 인간 연구의 결과는 일치하지 않습니다. 카이제곱 검정을 연습하려고 산모의 혼인 상태와 아기 성별의 독립성을 살펴보겠습니다.

남아와 여아의 산모들을 나눕니다.

In [114]:
male = live.query("babysex == 1")
female = live.query("babysex == 2")

열에는 아기 성별을, 행에는 산모의 혼인 상태 `fmarital`을 넣은 도수표를 만듭니다.

| 코드 | 혼인 상태 |
| --- | --- |
| 1 | 결혼 |
| 2 | 사별 |
| 3 | 이혼 |
| 4 | 별거 |
| 5 | 결혼한 적 없음 |

In [115]:
observed = pd.DataFrame()
observed["male"] = male["fmarital"].value_counts().sort_index()
observed["female"] = female["fmarital"].value_counts().sort_index()
observed

귀무가설은 성별에 따라 혼인 상태의 분포가 다르지 않다는 것입니다. 전체 자료의 혼인 상태 분포를 공통 확률로 사용합니다.

In [116]:
from empiricaldist import Pmf

pmf_fmarital = Pmf.from_seq(live["fmarital"])
pmf_fmarital

그 확률에 각 성별 집단의 인원 수를 곱해 기대 도수를 구합니다.

In [117]:
expected = pd.DataFrame()
expected["male"] = pmf_fmarital * observed["male"].sum()
expected["female"] = pmf_fmarital * observed["female"].sum()
expected

관측·기대 도수로 카이제곱 통계량을 계산하고 p값을 구하세요. 성별 두 범주와 혼인 상태 다섯 범주의 독립성 검정이므로 자유도는 $(5-1)(2-1)=4$입니다. `scipy.stats.chi2_contingency`와 비교할 수 있습니다. 결과가 트리버스–윌러드 가설을 어느 정도 뒷받침하는지, 상관과 인과를 구별하며 논의하세요.

> 번역자 주: 원문은 전체 셀 수에서 1을 뺀 자유도와 `chisquared`라는 함수명을 제시하지만, 공통 혼인 분포를 같은 자료에서 추정한 이 분할표에는 맞지 않습니다. `chisquare(..., axis=None)`를 사용하려면 기본 자유도 9에서 5를 더 빼도록 `ddof=5`로 조정해야 합니다. 단순 적합도 검정과 독립성 검정의 자유도를 구별하세요.

자유도와 모수 추정에 따른 보정은 SciPy의 [`chi2_contingency`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.contingency.chi2_contingency.html)와 [`chisquare`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.chisquare.html) 문서에서도 확인할 수 있습니다.

In [118]:
# Solution goes here

In [119]:
# Solution goes here

In [120]:
# Solution goes here

In [121]:
# Solution goes here

In [122]:
# Solution goes here

### 연습문제 14.4

두 집단 차이의 전후 변화를 나타내는 ‘차이의 차이’도 분석할 수 있습니다. 공학 팀 프로젝트에서 성별에 따른 작업 배분의 고정관념을 줄이려는 개입을 연구한 2014년 논문을 예로 보겠습니다.

Stein, Aragon, Moreno, Goodman(2014), 「Evidence for the persistent effects of an intervention to mitigate gender-stereotypical task allocation within student engineering teams」, *IEEE Frontiers in Education Conference*, 1–9쪽. [논문](https://ieeexplore.ieee.org/document/7044435/).

학생들은 프로젝트의 각 작업에 대한 자신의 기여를 7점 척도로 평가했습니다. 프로그래밍 작업에서 개입 전 남학생의 평균은 3.57(SE 0.28), 여학생은 1.91(SE 0.32)이었습니다. 개입 뒤에는 남학생 3.44(SE 0.16), 여학생 3.18(SE 0.16)으로 차이가 줄었습니다.

1. 네 평균 추정량의 표집분포를 나타내는 `Normal` 객체를 만드세요. 이미 표준오차가 있으므로 표집분포의 분산을 만들기 위해 표본 크기를 다시 알 필요는 없습니다.
2. 전후 각각의 남녀 평균 차이의 분포를 구하세요.
3. 두 차이의 차이, 즉 격차 변화의 분포를 구하고 95% 구간과 p값을 계산하세요.

이 계산에서 격차가 줄었다는 근거가 있나요?

> 번역자 주: 네 평균을 독립으로 놓았을 때의 계산이라는 가정을 명시하세요. 같은 학생이 전후에 응답했다면 공분산이 필요하므로, 제공된 네 표준오차만으로 정확한 변화의 분산을 정할 수 없습니다. 전후 격차 감소의 관찰과 개입의 인과효과 입증도 구분해야 합니다.

In [123]:
# Solution goes here

In [124]:
# Solution goes here

In [125]:
# Solution goes here

In [126]:
# Solution goes here

In [127]:
# Solution goes here

[Think Stats: Exploratory Data Analysis in Python, 3rd Edition](https://allendowney.github.io/ThinkStats/index.html)

Copyright 2024 [Allen B. Downey](https://allendowney.com)

코드 라이선스: [MIT](https://mit-license.org/)

본문·그림과 한국어 번역 라이선스: [CC BY-NC-SA 4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/)